In [3]:
import os

# Walk up until we find the project root
while not os.path.exists(os.path.join(os.getcwd(), "data")):
    os.chdir("..")

print(os.getcwd())  # should end in \causal-fairness-credit-scoring

e:\causal-fairness-credit-scoring\causal-fairness-credit-scoring


In [4]:
df = pd.read_csv("data/processed/cleaned_data.csv")
df.head()

,Age,Job,Credit amount,Duration,Sex_male,Housing_own,Housing_rent,Saving accounts_moderate,Saving accounts_quite rich,Saving accounts_rich,...,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others,Risk_good,Age_group_Senior,Age_group_Young
0,67,2,1169,6,True,True,False,False,False,False,...,False,False,False,False,True,False,False,True,True,False
1,22,2,5951,48,False,True,False,False,False,False,...,False,False,False,False,True,False,False,False,False,True
2,49,1,2096,12,True,True,False,False,False,False,...,False,False,True,False,False,False,False,True,True,False
3,45,2,7882,42,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
4,53,2,4870,24,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,True,False


In [5]:
target = "Risk"
sensitive_feature = "Sex"

In [7]:
print(df.columns.tolist())

['Age', 'Job', 'Credit amount', 'Duration', 'Sex_male', 'Housing_own', 'Housing_rent', 'Saving accounts_moderate', 'Saving accounts_quite rich', 'Saving accounts_rich', 'Checking account_moderate', 'Checking account_rich', 'Purpose_car', 'Purpose_domestic appliances', 'Purpose_education', 'Purpose_furniture/equipment', 'Purpose_radio/TV', 'Purpose_repairs', 'Purpose_vacation/others', 'Risk_good', 'Age_group_Senior', 'Age_group_Young']


In [11]:
print(df.columns.tolist())
print(df.shape)

['Age', 'Job', 'Credit amount', 'Duration', 'Sex_male', 'Housing_own', 'Housing_rent', 'Saving accounts_moderate', 'Saving accounts_quite rich', 'Saving accounts_rich', 'Checking account_moderate', 'Checking account_rich', 'Purpose_car', 'Purpose_domestic appliances', 'Purpose_education', 'Purpose_furniture/equipment', 'Purpose_radio/TV', 'Purpose_repairs', 'Purpose_vacation/others', 'Risk_good', 'Age_group_Senior', 'Age_group_Young']
(1000, 22)


In [12]:
# Reconstruct Risk from Risk_good (1 = good, 0 = bad)
df["Risk"] = df["Risk_good"].map({1: "good", 0: "bad"})
df.drop(columns=["Risk_good"], inplace=True)

print(df["Risk"].value_counts())

Series([], Name: count, dtype: int64)


In [13]:
target = "Risk"
X = df.drop(columns=[target])
y = df[target]
print(X.shape, y.shape)

(1000, 21) (1000,)


In [14]:
X = pd.get_dummies(X, drop_first=True)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

Series([], dtype: int64)


In [20]:
import numpy as np

print("NaN in X_train:", np.isnan(X_train).sum())
print("NaN in y_train:", np.isnan(y_train.astype(float)).sum() if hasattr(y_train, 'astype') else "check manually")
print("X_train dtype:", X_train.dtype)
print("X_train type:", type(X_train))

NaN in X_train: 0
NaN in y_train: 800
X_train dtype: float64
X_train type: <class 'numpy.ndarray'>


In [21]:
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Force to float
X_train = X_train.astype(float)
X_test = X_test.astype(float)

In [22]:
print(y_train.isnull().sum())
print(y_train.unique())

800
[nan]


In [23]:
# Before the train/test split, run this
print(X.dtypes[X.dtypes == 'object'])

Series([], dtype: object)


In [26]:
import pandas as pd
import numpy as np

# If X_train is a numpy array
print("Type:", type(X_train))
print("Shape:", X_train.shape)

# Convert back to DataFrame to inspect
X_train_df = pd.DataFrame(X_train)
print("\nNaN per column:")
print(X_train_df.isnull().sum()[X_train_df.isnull().sum() > 0])

print("\nInf values:", np.isinf(X_train_df.values).sum())
print("y_train NaN:", pd.Series(y_train).isnull().sum())
print("y_train unique:", pd.Series(y_train).unique())

Type: <class 'numpy.ndarray'>
Shape: (800, 21)

NaN per column:
Series([], dtype: int64)

Inf values: 0
y_train NaN: 800
y_train unique: [nan]


In [27]:
# Step 1 - Rebuild y from scratch
y = df["Risk_good"].map({1: "good", 0: "bad"}) if "Risk_good" in df.columns else df["Risk"]

print(y.isnull().sum())  # should be 0
print(y.unique())        # should be ['good', 'bad']

1000
[nan]


In [28]:
# Step 2 - Redo the split cleanly
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Risk", "Risk_good"], errors="ignore")
y = df["Risk_good"].map({1: "good", 0: "bad"}) if "Risk_good" in df.columns else df["Risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3 - Impute
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

print("y_train NaN:", pd.Series(y_train).isnull().sum())
print("y_train unique:", pd.Series(y_train).unique())

y_train NaN: 800
y_train unique: [nan]


In [ ]:
#visves